# ROGII - Wellbore Geology Prediction

Autoregressive LightGBM pipeline merged into a single notebook for Kaggle submission.

In [ ]:

from dataclasses import dataclass
from pathlib import Path
from typing import Iterable, Any
import math
import argparse
import sys
import warnings
import json
import os

import pandas as pd
import numpy as np
import joblib
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import GroupKFold, train_test_split
from tqdm.auto import tqdm


HORIZONTAL_SUFFIX = "__horizontal_well.csv"
TYPEWELL_SUFFIX = "__typewell.csv"


@dataclass(frozen=True)
class WellData:
    well_id: str
    split: str
    horizontal: pd.DataFrame
    typewell: pd.DataFrame
    horizontal_path: Path
    typewell_path: Path


def well_id_from_path(path: Path) -> str:
    name = path.name
    if "__" not in name:
        raise ValueError(f"Cannot parse well id from {path}")
    return name.split("__", 1)[0]


def split_dir(data_dir: str | Path, split: str) -> Path:
    path = Path(data_dir) / split
    if not path.exists():
        raise FileNotFoundError(f"Missing split directory: {path}")
    return path


def list_well_ids(data_dir: str | Path, split: str) -> list[str]:
    directory = split_dir(data_dir, split)
    ids = [well_id_from_path(path) for path in directory.glob(f"*{HORIZONTAL_SUFFIX}")]
    return sorted(ids)


def get_test_well_ids(data_dir: str | Path) -> list[str]:
    test_path = Path(data_dir) / "test"
    if not test_path.exists():
        return []
    return list_well_ids(data_dir, "test")


def read_horizontal(path: str | Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    df = df.copy()
    df["row_index"] = range(len(df))
    if "MD" in df.columns:
        df = df.sort_values("MD", kind="mergesort").reset_index(drop=True)
    return df


def read_typewell(path: str | Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    if "TVT" not in df.columns or "GR" not in df.columns:
        raise ValueError(f"Typewell file must contain TVT and GR columns: {path}")
    return df.sort_values("TVT", kind="mergesort").reset_index(drop=True)


def load_well(data_dir: str | Path, split: str, well_id: str) -> WellData:
    directory = split_dir(data_dir, split)
    horizontal_path = directory / f"{well_id}{HORIZONTAL_SUFFIX}"
    typewell_path = directory / f"{well_id}{TYPEWELL_SUFFIX}"
    if not horizontal_path.exists():
        raise FileNotFoundError(horizontal_path)
    if not typewell_path.exists():
        raise FileNotFoundError(typewell_path)
    return WellData(
        well_id=well_id,
        split=split,
        horizontal=read_horizontal(horizontal_path),
        typewell=read_typewell(typewell_path),
        horizontal_path=horizontal_path,
        typewell_path=typewell_path,
    )


def load_wells(
    data_dir: str | Path,
    split: str,
    well_ids: Iterable[str] | None = None,
) -> list[WellData]:
    ids = list(well_ids) if well_ids is not None else list_well_ids(data_dir, split)
    return [load_well(data_dir, split, well_id) for well_id in ids]


def read_submission_template(data_dir: str | Path) -> pd.DataFrame:
    path = Path(data_dir) / "sample_submission.csv"
    if not path.exists():
        raise FileNotFoundError(path)
    sub = pd.read_csv(path)
    required = {"id", "tvt"}
    missing = required.difference(sub.columns)
    if missing:
        raise ValueError(f"sample_submission.csv missing columns: {sorted(missing)}")
    return sub

BASE_INPUT_COLUMNS = ["MD", "X", "Y", "Z", "GR"]
HISTORY_COLUMNS = ["TVT", "X", "Y", "Z", "GR"]
LAG_STEPS = [1, 2, 3, 5, 10, 20, 50, 100]
ROLLING_WINDOWS = [5, 10, 20, 50, 100]
TYPEWELL_WINDOWS = [5, 10, 20]


@dataclass(frozen=True)
class TypewellIndex:
    tvt: np.ndarray
    gr: np.ndarray


def prepare_typewell_index(typewell: pd.DataFrame) -> TypewellIndex:
    tw = typewell[["TVT", "GR"]].dropna(subset=["TVT"]).sort_values("TVT")
    return TypewellIndex(
        tvt=tw["TVT"].to_numpy(dtype=float),
        gr=tw["GR"].to_numpy(dtype=float),
    )


def _safe_numeric(series: pd.Series) -> pd.Series:
    return pd.to_numeric(series, errors="coerce")


def prepare_static_features(horizontal: pd.DataFrame) -> pd.DataFrame:
    df = horizontal.copy()
    out = pd.DataFrame(index=df.index)
    out["row_position"] = np.arange(len(df), dtype=float)

    for col in BASE_INPUT_COLUMNS:
        out[col] = _safe_numeric(df[col]) if col in df.columns else np.nan

    out["md_from_start"] = out["MD"] - out["MD"].iloc[0]
    out["dX"] = out["X"].diff()
    out["dY"] = out["Y"].diff()
    out["dZ"] = out["Z"].diff()
    out["dMD"] = out["MD"].diff()
    out["xy_step_distance"] = np.sqrt(out["dX"] ** 2 + out["dY"] ** 2)
    out["step_distance"] = np.sqrt(out["dX"] ** 2 + out["dY"] ** 2 + out["dZ"] ** 2)
    out["path_distance"] = out["step_distance"].fillna(0).cumsum()
    out["dZ_dMD"] = out["dZ"] / out["dMD"].replace(0, np.nan)
    
    out["GR_diff"] = out["GR"].diff().fillna(0)
    
    out["GR_static_mean_10"] = out["GR"].rolling(10, min_periods=1).mean().fillna(0)
    out["GR_static_std_10"] = out["GR"].rolling(10, min_periods=2).std().fillna(0)
    out["GR_static_mean_50"] = out["GR"].rolling(50, min_periods=1).mean().fillna(0)
    out["GR_static_std_50"] = out["GR"].rolling(50, min_periods=2).std().fillna(0)
    
    out["GR_lookahead_10"] = out["GR"].shift(-10).fillna(0)
    out["GR_lookahead_30"] = out["GR"].shift(-30).fillna(0)
    
    out["GR_lookback_10"] = out["GR"].shift(10).fillna(0)
    out["GR_lookback_30"] = out["GR"].shift(30).fillna(0)
    
    return out


def _nearest_indices(typewell_index: TypewellIndex, expected_tvt: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    tvt = typewell_index.tvt
    nearest = np.full(len(expected_tvt), -1, dtype=int)
    gap = np.full(len(expected_tvt), np.nan, dtype=float)
    if len(tvt) == 0:
        return nearest, gap

    valid = np.isfinite(expected_tvt)
    pos = np.searchsorted(tvt, expected_tvt[valid], side="left")
    right = np.clip(pos, 0, len(tvt) - 1)
    left = np.clip(pos - 1, 0, len(tvt) - 1)
    use_left = np.abs(expected_tvt[valid] - tvt[left]) <= np.abs(expected_tvt[valid] - tvt[right])
    chosen = np.where(use_left, left, right)
    nearest[np.where(valid)[0]] = chosen
    gap[np.where(valid)[0]] = expected_tvt[valid] - tvt[chosen]
    return nearest, gap


def _typewell_window_stats(typewell_index: TypewellIndex, nearest: np.ndarray, window: int) -> tuple[np.ndarray, np.ndarray]:
    mean = np.full(len(nearest), np.nan, dtype=float)
    std = np.full(len(nearest), np.nan, dtype=float)
    gr = typewell_index.gr
    if len(gr) == 0:
        return mean, std

    half = window // 2
    for row, idx in enumerate(nearest):
        if idx < 0:
            continue
        start = max(0, idx - half)
        end = min(len(gr), idx + half + 1)
        values = gr[start:end]
        if np.isfinite(values).any():
            mean[row] = np.nanmean(values)
            finite_count = np.isfinite(values).sum()
            std[row] = np.nanstd(values, ddof=1) if finite_count > 1 else np.nan
    return mean, std


def make_typewell_features(
    current_gr: pd.Series,
    expected_tvt: pd.Series,
    typewell_index: TypewellIndex,
) -> pd.DataFrame:
    expected = expected_tvt.to_numpy(dtype=float)
    nearest, gap = _nearest_indices(typewell_index, expected)
    out = pd.DataFrame(index=expected_tvt.index)
    out["expected_tvt"] = expected_tvt

    nearest_gr = np.full(len(expected), np.nan, dtype=float)
    valid = nearest >= 0
    if len(typewell_index.gr):
        nearest_gr[valid] = typewell_index.gr[nearest[valid]]
    out["typewell_gr_at_nearest_tvt"] = nearest_gr
    out["gr_minus_typewell_gr"] = current_gr.to_numpy(dtype=float) - nearest_gr
    out["nearest_typewell_tvt_gap"] = gap

    for window in TYPEWELL_WINDOWS:
        mean, std = _typewell_window_stats(typewell_index, nearest, window)
        out[f"typewell_gr_roll_mean_{window}"] = mean
        out[f"typewell_gr_roll_std_{window}"] = std
    return out


def build_feature_frame(
    horizontal: pd.DataFrame,
    typewell: pd.DataFrame,
    tvt_history: pd.Series | np.ndarray,
    is_training: bool = False,
) -> pd.DataFrame:
    static = prepare_static_features(horizontal)
    history = pd.DataFrame(index=horizontal.index)
    tvt_series = pd.Series(tvt_history, index=horizontal.index, dtype=float)
    history["TVT"] = tvt_series.copy()
    if is_training:
        history["TVT"] += np.random.normal(0, 1.0, size=len(history["TVT"]))
        
    for col in ["X", "Y", "Z", "GR"]:
        history[col] = _safe_numeric(horizontal[col]) if col in horizontal.columns else np.nan

    parts = [static]
    for col in HISTORY_COLUMNS:
        for lag in LAG_STEPS:
            parts.append(history[col].shift(lag).rename(f"{col}_lag_{lag}"))
        shifted = history[col].shift(1)
        for window in ROLLING_WINDOWS:
            parts.append(shifted.rolling(window, min_periods=1).mean().rename(f"{col}_roll_mean_{window}"))
            parts.append(shifted.rolling(window, min_periods=2).std().rename(f"{col}_roll_std_{window}"))

    typewell_index = prepare_typewell_index(typewell)
    expected_tvt = tvt_series.shift(1)
    parts.append(make_typewell_features(static["GR"], expected_tvt, typewell_index))
    
    start_tvt = tvt_series.iloc[0]
    current_drift = expected_tvt - start_tvt
    parts.append(current_drift.rename("Current_Drift"))
    
    return pd.concat(parts, axis=1)


def target_mask(horizontal: pd.DataFrame, scope: str = "prediction") -> pd.Series:
    if "TVT" not in horizontal.columns:
        return pd.Series(False, index=horizontal.index)
    mask = horizontal["TVT"].notna()
    if scope == "prediction":
        if "TVT_input" not in horizontal.columns:
            raise ValueError("prediction scope requires TVT_input")
        mask &= horizontal["TVT_input"].isna()
    elif scope == "known":
        if "TVT_input" not in horizontal.columns:
            raise ValueError("known scope requires TVT_input")
        mask &= horizontal["TVT_input"].notna()
    elif scope != "all":
        raise ValueError(f"Unknown train scope: {scope}")
    return mask


def build_training_matrix(
    horizontal: pd.DataFrame,
    typewell: pd.DataFrame,
    scope: str = "prediction",
) -> tuple[pd.DataFrame, pd.Series]:
    if "TVT" not in horizontal.columns:
        raise ValueError("Training horizontal data must contain TVT")
    features = build_feature_frame(horizontal, typewell, horizontal["TVT"], is_training=True)
    
    delta_tvt = horizontal["TVT"].diff().fillna(0)
    
    mask = target_mask(horizontal, scope=scope)
    mask &= delta_tvt.notna()
    
    return features.loc[mask].reset_index(drop=True), delta_tvt.loc[mask].reset_index(drop=True)


def _array_stats(values: np.ndarray) -> tuple[float, float]:
    total = 0.0
    count = 0
    for val in values:
        if math.isfinite(val):
            total += val
            count += 1
            
    if count == 0:
        return np.nan, np.nan
        
    mean = total / count
    if count == 1:
        return mean, np.nan
        
    sq_diff = 0.0
    for val in values:
        if math.isfinite(val):
            sq_diff += (val - mean) ** 2
            
    std = math.sqrt(sq_diff / (count - 1))
    return mean, std

def _rolling_stats(values: np.ndarray, idx: int, window: int) -> tuple[float, float]:
    start = max(0, idx - window)
    hist = values[start:idx]
    return _array_stats(hist)


def make_single_row_features(
    static_features: pd.DataFrame,
    horizontal: pd.DataFrame,
    typewell_index: TypewellIndex,
    tvt_work: np.ndarray,
    idx: int,
    start_tvt: float,
) -> pd.DataFrame:
    row = static_features.iloc[idx].to_dict()
    history_arrays = {
        "TVT": tvt_work.astype(float, copy=False),
        "X": horizontal["X"].to_numpy(dtype=float),
        "Y": horizontal["Y"].to_numpy(dtype=float),
        "Z": horizontal["Z"].to_numpy(dtype=float),
        "GR": horizontal["GR"].to_numpy(dtype=float),
    }

    for col, values in history_arrays.items():
        for lag in LAG_STEPS:
            row[f"{col}_lag_{lag}"] = values[idx - lag] if idx - lag >= 0 else np.nan
        for window in ROLLING_WINDOWS:
            mean, std = _rolling_stats(values, idx, window)
            row[f"{col}_roll_mean_{window}"] = mean
            row[f"{col}_roll_std_{window}"] = std

    expected = tvt_work[idx - 1] if idx > 0 else np.nan
    row["Current_Drift"] = expected - start_tvt
    
    current_gr = row.get("GR", np.nan)
    tw_features = make_typewell_features(
        pd.Series([current_gr]),
        pd.Series([expected]),
        typewell_index,
    )
    row.update(tw_features.iloc[0].to_dict())
    return pd.DataFrame([row])


class InferenceFeatureBuilder:
    def __init__(self, static_features: pd.DataFrame, horizontal: pd.DataFrame, typewell_index: TypewellIndex, feature_columns: list[str], start_tvt: float):
        self.static_dict = {col: static_features[col].to_numpy() for col in static_features.columns}
        self.history_arrays = {
            "X": horizontal["X"].to_numpy(dtype=float) if "X" in horizontal.columns else np.full(len(horizontal), np.nan),
            "Y": horizontal["Y"].to_numpy(dtype=float) if "Y" in horizontal.columns else np.full(len(horizontal), np.nan),
            "Z": horizontal["Z"].to_numpy(dtype=float) if "Z" in horizontal.columns else np.full(len(horizontal), np.nan),
            "GR": horizontal["GR"].to_numpy(dtype=float) if "GR" in horizontal.columns else np.full(len(horizontal), np.nan),
        }
        self.typewell_index = typewell_index
        self.feature_columns = feature_columns
        self.tw_tvt = typewell_index.tvt
        self.tw_gr = typewell_index.gr
        self.start_tvt = start_tvt
        
    def build_row(self, tvt_work: np.ndarray, idx: int) -> np.ndarray:
        row = {}
        for col, arr in self.static_dict.items():
            row[col] = arr[idx]
            
        self.history_arrays["TVT"] = tvt_work
        
        for col in ["TVT", "X", "Y", "Z", "GR"]:
            values = self.history_arrays[col]
            for lag in LAG_STEPS:
                row[f"{col}_lag_{lag}"] = values[idx - lag] if idx - lag >= 0 else np.nan
            for window in ROLLING_WINDOWS:
                mean, std = _rolling_stats(values, idx, window)
                row[f"{col}_roll_mean_{window}"] = mean
                row[f"{col}_roll_std_{window}"] = std
                
        expected = tvt_work[idx - 1] if idx > 0 else np.nan
        row["Current_Drift"] = expected - self.start_tvt
        
        current_gr = row.get("GR", np.nan)
        
        if np.isfinite(expected) and len(self.tw_tvt) > 0:
            pos = np.searchsorted(self.tw_tvt, expected, side="left")
            right = np.clip(pos, 0, len(self.tw_tvt) - 1)
            left = np.clip(pos - 1, 0, len(self.tw_tvt) - 1)
            if abs(expected - self.tw_tvt[left]) <= abs(expected - self.tw_tvt[right]):
                nearest = left
            else:
                nearest = right
            
            nearest_gr = self.tw_gr[nearest]
            row["expected_tvt"] = expected
            row["typewell_gr_at_nearest_tvt"] = nearest_gr
            row["gr_minus_typewell_gr"] = current_gr - nearest_gr
            row["nearest_typewell_tvt_gap"] = expected - self.tw_tvt[nearest]
            
            for window in TYPEWELL_WINDOWS:
                half = window // 2
                start = max(0, nearest - half)
                end = min(len(self.tw_gr), nearest + half + 1)
                values = self.tw_gr[start:end]
                mean, std = _array_stats(values)
                row[f"typewell_gr_roll_mean_{window}"] = mean
                row[f"typewell_gr_roll_std_{window}"] = std
        else:
            row["expected_tvt"] = expected
            row["typewell_gr_at_nearest_tvt"] = np.nan
            row["gr_minus_typewell_gr"] = np.nan
            row["nearest_typewell_tvt_gap"] = np.nan
            for window in TYPEWELL_WINDOWS:
                row[f"typewell_gr_roll_mean_{window}"] = np.nan
                row[f"typewell_gr_roll_std_{window}"] = np.nan
                
        out = np.zeros((1, len(self.feature_columns)), dtype=float)
        for i, col in enumerate(self.feature_columns):
            out[0, i] = row.get(col, np.nan)
        return out



def ensure_feature_columns(frame: pd.DataFrame, feature_columns: Iterable[str]) -> pd.DataFrame:
    columns = list(feature_columns)
    aligned = frame.copy()
    for col in columns:
        if col not in aligned.columns:
            aligned[col] = np.nan
    return aligned[columns]

MODEL_FILENAME = "model.joblib"
FEATURES_FILENAME = "feature_columns.json"
CONFIG_FILENAME = "config.json"
METRICS_FILENAME = "metrics.json"


def get_lightgbm_regressor(random_state: int = 42, **overrides: Any):
    try:
        from lightgbm import LGBMRegressor
    except ImportError as exc:
        raise ImportError("LightGBM is required. Install it with: pip install lightgbm") from exc

    params = {
        "objective": "huber",
        "metric": "rmse",
        "n_estimators": 2000,
        "learning_rate": 0.03,
        "num_leaves": 63,
        "max_depth": -1,
        "min_child_samples": 50,
        "subsample": 0.9,
        "subsample_freq": 1,
        "colsample_bytree": 0.9,
        "reg_alpha": 0.1,
        "reg_lambda": 0.5,
        "random_state": random_state,
        "n_jobs": -1,
        "verbosity": -1,
    }
    params.update({key: value for key, value in overrides.items() if value is not None})
    return LGBMRegressor(**params)


def save_artifacts(
    model: Any,
    model_dir: str | Path,
    feature_columns: list[str],
    config: dict[str, Any],
    metrics: dict[str, Any],
) -> None:
    path = Path(model_dir)
    path.mkdir(parents=True, exist_ok=True)
    joblib.dump(model, path / MODEL_FILENAME)
    (path / FEATURES_FILENAME).write_text(json.dumps(feature_columns, indent=2), encoding="utf-8")
    (path / CONFIG_FILENAME).write_text(json.dumps(config, indent=2, default=str), encoding="utf-8")
    (path / METRICS_FILENAME).write_text(json.dumps(metrics, indent=2, default=str), encoding="utf-8")


def load_artifacts(model_dir: str | Path):
    path = Path(model_dir)
    model = joblib.load(path / MODEL_FILENAME)
    feature_columns = json.loads((path / FEATURES_FILENAME).read_text(encoding="utf-8"))
    config = json.loads((path / CONFIG_FILENAME).read_text(encoding="utf-8"))
    return model, feature_columns, config

def rmse(y_true, y_pred) -> float:
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


def eligible_train_ids(data_dir: str | Path, exclude_test_ids: bool = True) -> list[str]:
    ids = list_well_ids(data_dir, "train")
    if exclude_test_ids:
        test_ids = set(get_test_well_ids(data_dir))
        ids = [well_id for well_id in ids if well_id not in test_ids]
    return ids


def build_matrix_for_wells(
    data_dir: str | Path,
    well_ids: list[str],
    scope: str,
    max_rows_per_well: int | None = None,
) -> tuple[pd.DataFrame, pd.Series, np.ndarray]:
    x_parts: list[pd.DataFrame] = []
    y_parts: list[pd.Series] = []
    groups: list[np.ndarray] = []

    for well_id in tqdm(well_ids, desc="Building features"):
        well = load_well(data_dir, "train", well_id)
        x_well, y_well = build_training_matrix(well.horizontal, well.typewell, scope=scope)
        if max_rows_per_well is not None and len(x_well) > max_rows_per_well:
            x_well = x_well.tail(max_rows_per_well).reset_index(drop=True)
            y_well = y_well.tail(max_rows_per_well).reset_index(drop=True)
        x_parts.append(x_well)
        y_parts.append(y_well)
        groups.append(np.repeat(well_id, len(x_well)))

    if not x_parts:
        raise ValueError("No training wells were selected.")
    x = pd.concat(x_parts, ignore_index=True)
    y = pd.concat(y_parts, ignore_index=True)
    group_values = np.concatenate(groups)
    return x, y, group_values


def recursive_validate(
    model,
    feature_columns: list[str],
    data_dir: str | Path,
    well_ids: list[str],
    max_wells: int | None = None,
) -> dict[str, Any]:
    if max_wells is not None and max_wells <= 0:
        return {
            "recursive_rmse": None,
            "rows": 0,
            "wells": [],
            "skipped_wells": well_ids,
            "by_well": {},
        }
    selected_ids = well_ids[:max_wells] if max_wells is not None else well_ids
    rows: list[pd.DataFrame] = []
    for well_id in tqdm(selected_ids, desc="Recursive validation"):
        well = load_well(data_dir, "train", well_id)
        pred_df, _ = recursive_predict_well(model, feature_columns, well)
        truth = well.horizontal[["row_index", "TVT"]].copy()
        truth["id"] = truth["row_index"].map(lambda idx: f"{well_id}_{int(idx)}")
        merged = pred_df.merge(truth[["id", "TVT"]], on="id", how="left")
        merged["well_id"] = well_id
        rows.append(merged)

    if not rows:
        return {"recursive_rmse": None, "rows": 0, "by_well": {}}
    frame = pd.concat(rows, ignore_index=True)
    score = rmse(frame["TVT"], frame["tvt"])
    by_well = {
        well_id: rmse(group["TVT"], group["tvt"])
        for well_id, group in frame.groupby("well_id")
    }
    return {
        "recursive_rmse": score,
        "rows": int(len(frame)),
        "wells": selected_ids,
        "skipped_wells": well_ids[len(selected_ids) :],
        "by_well": by_well,
    }


def fit_lgbm(
    x_train: pd.DataFrame,
    y_train: pd.Series,
    x_valid: pd.DataFrame | None,
    y_valid: pd.Series | None,
    args: argparse.Namespace,
):
    model = get_lightgbm_regressor(
        random_state=args.random_state,
        n_estimators=args.n_estimators,
        learning_rate=args.learning_rate,
        num_leaves=args.num_leaves,
    )
    fit_kwargs: dict[str, Any] = {}
    if x_valid is not None and y_valid is not None and len(x_valid):
        try:
            from lightgbm import early_stopping, log_evaluation

            fit_kwargs["eval_set"] = [(x_valid, y_valid)]
            fit_kwargs["eval_metric"] = "rmse"
            fit_kwargs["callbacks"] = [
                early_stopping(args.early_stopping_rounds, verbose=False),
                log_evaluation(args.log_period),
            ]
        except ImportError:
            pass
    model.fit(x_train, y_train, **fit_kwargs)
    return model


def run_holdout(args: argparse.Namespace, all_ids: list[str]) -> tuple[Any, list[str], dict[str, Any]]:
    train_ids, valid_ids = train_test_split(
        all_ids,
        test_size=args.valid_size,
        random_state=args.random_state,
        shuffle=True,
    )
    if args.max_wells is not None:
        train_ids = train_ids[: max(1, int(args.max_wells * (1 - args.valid_size)))]
        valid_ids = valid_ids[: max(1, args.max_wells - len(train_ids))]

    x_train, y_train, _ = build_matrix_for_wells(
        args.data_dir,
        train_ids,
        args.train_scope,
        max_rows_per_well=args.max_train_rows_per_well,
    )
    x_valid, y_valid, _ = build_matrix_for_wells(
        args.data_dir,
        valid_ids,
        args.train_scope,
        max_rows_per_well=args.max_train_rows_per_well,
    )
    feature_columns = list(x_train.columns)
    x_valid = ensure_feature_columns(x_valid, feature_columns)
    model = fit_lgbm(x_train, y_train, x_valid, y_valid, args)

    valid_pred = model.predict(x_valid)
    metrics = {
        "split": "holdout",
        "train_wells": train_ids,
        "valid_wells": valid_ids,
        "train_rows": int(len(x_train)),
        "valid_rows": int(len(x_valid)),
        "teacher_forced_rmse": rmse(y_valid, valid_pred),
        "recursive_validation": recursive_validate(
            model,
            feature_columns,
            args.data_dir,
            valid_ids,
            max_wells=args.recursive_valid_wells,
        ),
    }
    return model, feature_columns, metrics


def run_groupkfold(args: argparse.Namespace, all_ids: list[str]) -> tuple[Any, list[str], dict[str, Any]]:
    ids = all_ids[: args.max_wells] if args.max_wells is not None else all_ids
    x_all, y_all, groups = build_matrix_for_wells(
        args.data_dir,
        ids,
        args.train_scope,
        max_rows_per_well=args.max_train_rows_per_well,
    )
    feature_columns = list(x_all.columns)
    splitter = GroupKFold(n_splits=args.n_splits)
    fold_metrics = []

    for fold, (train_idx, valid_idx) in enumerate(splitter.split(x_all, y_all, groups), start=1):
        print(f"Training fold {fold}/{args.n_splits}")
        x_train, y_train = x_all.iloc[train_idx], y_all.iloc[train_idx]
        x_valid, y_valid = x_all.iloc[valid_idx], y_all.iloc[valid_idx]
        model = fit_lgbm(x_train, y_train, x_valid, y_valid, args)
        pred = model.predict(x_valid)
        
        valid_wells = sorted(set(groups[valid_idx]))
        fold_metrics.append(
            {
                "fold": fold,
                "valid_wells": valid_wells,
                "train_rows": len(train_idx),
                "valid_rows": int(len(valid_idx)),
                "teacher_forced_rmse": rmse(y_valid, pred),
            }
        )

    print("Training final model on all selected non-test wells")
    final_model = fit_lgbm(x_all, y_all, None, None, args)
    scores = [item["teacher_forced_rmse"] for item in fold_metrics]
    metrics = {
        "split": "groupkfold",
        "n_splits": args.n_splits,
        "wells": ids,
        "rows": int(len(x_all)),
        "folds": fold_metrics,
        "mean_teacher_forced_rmse": float(np.mean(scores)),
        "std_teacher_forced_rmse": float(np.std(scores)),
    }
    return final_model, feature_columns, metrics


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="Train autoregressive LightGBM for ROGII TVT prediction.")
    parser.add_argument("--data_dir", type=str, default="data")
    parser.add_argument("--exp_name", type=str, default="lgbm_baseline")
    parser.add_argument("--split", choices=["holdout", "groupkfold"], default="holdout")
    parser.add_argument("--valid_size", type=float, default=0.2)
    parser.add_argument("--n_splits", type=int, default=5)
    parser.add_argument("--train_scope", choices=["prediction", "all", "known"], default="prediction")
    parser.add_argument("--include_test_ids", action="store_true", help="Allow leakage-prone training on current test IDs.")
    parser.add_argument("--max_wells", type=int, default=None, help="Debug: limit number of train wells.")
    parser.add_argument("--max_train_rows_per_well", type=int, default=None, help="Debug: keep only the tail rows per well.")
    parser.add_argument("--random_state", type=int, default=42)
    parser.add_argument("--n_estimators", type=int, default=2000)
    parser.add_argument("--learning_rate", type=float, default=0.03)
    parser.add_argument("--num_leaves", type=int, default=63)
    parser.add_argument("--early_stopping_rounds", type=int, default=100)
    parser.add_argument("--log_period", type=int, default=100)
    parser.add_argument(
        "--recursive_valid_wells",
        type=int,
        default=3,
        help="Number of validation wells for slow recursive validation. Use 0 to skip.",
    )
    return parser.parse_args()

# Add src to sys.path to prevent shadowing from global packages



def get_start_coords(horizontal: pd.DataFrame) -> tuple[float, float]:
    valid_xy = horizontal.dropna(subset=["X", "Y"])
    if len(valid_xy) == 0:
        return np.nan, np.nan
    return float(valid_xy.iloc[0]["X"]), float(valid_xy.iloc[0]["Y"])

def find_offset_wells(test_well, train_wells, k=3):
    test_x, test_y = get_start_coords(test_well.horizontal)
    if np.isnan(test_x) or np.isnan(test_y):
        return []

    distances = []
    for tw in train_wells:
        if tw.well_id == test_well.well_id:
            continue
        tw_x, tw_y = get_start_coords(tw.horizontal)
        if np.isnan(tw_x) or np.isnan(tw_y):
            continue
        dist = np.sqrt((tw_x - test_x)**2 + (tw_y - test_y)**2)
        distances.append((dist, tw))
        
    distances.sort(key=lambda x: x[0])
    return [tw for _, tw in distances[:k]]

def recursive_predict_well(model, feature_columns: list[str], well, train_wells=None) -> tuple[pd.DataFrame, np.ndarray]:
    horizontal = well.horizontal.reset_index(drop=True)
    static_features = prepare_static_features(horizontal)
    typewell_index = prepare_typewell_index(well.typewell)
    tvt_work = horizontal["TVT_input"].to_numpy(dtype=float).copy()
    row_indices = horizontal["row_index"].to_numpy(dtype=int)

    start_tvt_idx = horizontal["TVT_input"].last_valid_index()
    start_tvt = float(horizontal.loc[start_tvt_idx, "TVT_input"]) if start_tvt_idx is not None else 0.0

    builder = InferenceFeatureBuilder(static_features, horizontal, typewell_index, feature_columns, start_tvt)

    offset_wells = []
    if train_wells is not None:
        offset_wells = find_offset_wells(well, train_wells, k=3)

    test_md = horizontal["MD"].to_numpy(dtype=float)
    regional_dip_array = np.full(len(horizontal), np.nan)

    if offset_wells:
        interpolated_dips = []
        for ow in offset_wells:
            ow_horz = ow.horizontal
            if "MD" in ow_horz.columns and "TVT" in ow_horz.columns:
                delta_tvt = ow_horz["TVT"].diff().fillna(0).to_numpy(dtype=float)
                ow_md = ow_horz["MD"].to_numpy(dtype=float)
                valid_mask = np.isfinite(ow_md) & np.isfinite(delta_tvt)
                if valid_mask.sum() > 1:
                    ow_md_valid = ow_md[valid_mask]
                    delta_tvt_valid = delta_tvt[valid_mask]
                    sort_idx = np.argsort(ow_md_valid)
                    ow_md_valid = ow_md_valid[sort_idx]
                    delta_tvt_valid = delta_tvt_valid[sort_idx]
                    
                    interp_dip = np.interp(test_md, ow_md_valid, delta_tvt_valid, left=np.nan, right=np.nan)
                    interpolated_dips.append(interp_dip)
                    
        if interpolated_dips:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore", category=RuntimeWarning)
                regional_dip_array = np.nanmean(np.vstack(interpolated_dips), axis=0)

    records: list[dict[str, float | str]] = []
    for idx in range(len(horizontal)):
        if np.isfinite(tvt_work[idx]):
            continue
        
        regional_dip = regional_dip_array[idx]
        
        features_array = builder.build_row(tvt_work, idx)
        predicted_delta = float(model.booster_.predict(features_array)[0])
        
        if np.isfinite(regional_dip):
            clipped_delta = float(np.clip(predicted_delta, regional_dip - 0.15, regional_dip + 0.15))
        else:
            clipped_delta = float(np.clip(predicted_delta, -0.3, 0.3))
        
        new_tvt = tvt_work[idx - 1] + clipped_delta
        tvt_work[idx] = new_tvt
        records.append({"id": f"{well.well_id}_{row_indices[idx]}", "tvt": new_tvt})

    return pd.DataFrame(records), tvt_work



def rmse(y_true: pd.Series, y_pred: pd.Series) -> float:
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))


def read_submission_template(data_dir: str | Path) -> pd.DataFrame:
    path = Path(data_dir) / "sample_submission.csv"
    if not path.exists():
        raise FileNotFoundError(f"Missing submission template: {path}")
    return pd.read_csv(path)

def predict_submission(data_dir: str | Path, model_dir: str | Path, output: str | Path) -> pd.DataFrame:
    model, feature_columns, _ = load_artifacts(model_dir)
    template = read_submission_template(data_dir)
    test_ids = get_test_well_ids(data_dir)

    predictions = []
    for well_id in tqdm(test_ids, desc="Predicting test wells"):
        well = load_well(data_dir, "test", well_id)
        pred_df, _ = recursive_predict_well(model, feature_columns, well)
        predictions.append(pred_df)

    pred = pd.concat(predictions, ignore_index=True) if predictions else pd.DataFrame(columns=["id", "tvt"])
    merged = template[["id"]].merge(pred, on="id", how="left", validate="one_to_one")
    if merged["tvt"].isna().any():
        missing = merged.loc[merged["tvt"].isna(), "id"].head(10).tolist()
        raise ValueError(f"Missing predictions for submission ids: {missing}")

    output = Path(output)
    output.parent.mkdir(parents=True, exist_ok=True)
    merged.to_csv(output, index=False)
    return merged

# Adapt directories for Kaggle
import os
if os.path.exists("/kaggle/input/competitions/rogii-wellbore-geology-prediction"):
    DATA_DIR = "/kaggle/input/competitions/rogii-wellbore-geology-prediction"
elif os.path.exists("/kaggle/input/rogii-wellbore-geology-prediction"):
    DATA_DIR = "/kaggle/input/rogii-wellbore-geology-prediction"
else:
    DATA_DIR = "data"

OUTPUT_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "model"
SUBMISSION_PATH = "submission.csv"

class Args:
    data_dir = DATA_DIR
    train_scope = "prediction"
    valid_size = 0.2
    random_state = 42
    n_estimators = 2000
    learning_rate = 0.03
    num_leaves = 63
    early_stopping_rounds = 100
    log_period = 100
    max_train_rows_per_well = None
    max_wells = None
    n_splits = 5
    recursive_valid_wells = 0
    include_test_ids = False
    exp_name = "kaggle_run"

def run_kaggle_pipeline():
    args = Args()
    
    all_ids = eligible_train_ids(args.data_dir, exclude_test_ids=not args.include_test_ids)
    print(f"Selected train wells: {len(all_ids)}")
    
    # Holdout validation (default in the project)
    model, feature_columns, metrics = run_holdout(args, all_ids)
    
    model_dir = Path(OUTPUT_DIR) / args.exp_name
    save_artifacts(model, model_dir, feature_columns, vars(args), metrics)
    print(f"Saved model artifacts to {model_dir}")
    print(f"Metrics: {metrics}")
    
    # Predict submission
    submission = predict_submission(args.data_dir, model_dir, SUBMISSION_PATH)
    print(f"Saved submission to {SUBMISSION_PATH}")

if __name__ == "__main__":
    run_kaggle_pipeline()
